# 19. GAN과 Diffusion — 흐릿함을 넘어서

> **제19장** · **이론편 대응: 16장 (생성 모델 II)**
> **예상 소요**: 70분
> **필요 사양**: **[CPU]** 로 실행 가능 (GPU 있으면 더 큰 규모 가능)
> **추가 설치**: 없음
> **다운로드**: FashionMNIST (15장에서 받았다면 재사용)

---

## 이 장에서 하는 일

18장에서 VAE가 만든 이미지가 흐릿했다. 그 문제를 다르게 푼 두 접근을 다룬다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | GAN의 발상 — 두 신경망의 경쟁 | 16.1절 |
| 2 | GAN 구현과 학습 | 16.2절 |
| 3 | **학습 불안정성 직접 확인** | 16.2절 |
| 4 | **노이즈 누적표 검증** ★ | 16.3절 |
| 5 | Diffusion 직접 구현 | 16.3절 |
| 6 | 세 모델 비교 | 16.4절 |

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
from pathlib import Path

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} / 장치: {device}")

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent

train_data = datasets.FashionMNIST(
    root=str(root / "data"), train=True, download=True,
    transform=transforms.ToTensor())

N_TRAIN = 4000
train_loader = DataLoader(Subset(train_data, range(N_TRAIN)),
                          batch_size=128, shuffle=True)
print(f"학습 데이터: {N_TRAIN:,}장")

---

## 1. GAN의 발상 — 이론편 16.1절

18장에서 본 흐릿함의 원인은 **픽셀 단위 오차를 최소화했기 때문**이었다.
애매한 곳에 평균값을 칠하는 것이 오차 면에서 안전하니까.

GAN은 이 손실함수 자체를 바꾼다. **"진짜처럼 보이는가"를 다른 신경망이 판단하게 하는 것**이다.

| 역할 | 하는 일 | 목표 |
|---|---|---|
| 생성자(Generator) | 무작위 벡터 → 이미지 | 판별자를 속이기 |
| 판별자(Discriminator) | 이미지 → 진짜/가짜 | 정확히 구별하기 |

위조범과 감별사의 경쟁에 비유된다. 둘이 함께 실력이 늘어가는 구조다.

**평균값을 칠하면 판별자에게 바로 들킨다.** 그래서 생성자는 선명한 것을 만들어야 한다.

In [ ]:
import torch
import torch.nn as nn


LATENT_DIM = 32


class Generator(nn.Module):
    # 무작위 벡터를 이미지로 바꾸는 신경망 (이론편 16.1절)

    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid(),               # 픽셀값 0~1
        )

    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)


class Discriminator(nn.Module):
    # 이미지가 진짜인지 판단하는 신경망

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 1),          # 로짓 하나 (진짜일 확률)
        )

    def forward(self, x):
        return self.net(x)


G = Generator().to(device)
D = Discriminator().to(device)

print("=" * 55)
print("GAN 구성")
print("=" * 55)
print(f"생성자 입력  : 무작위 벡터 {LATENT_DIM}차원")
print(f"생성자 출력  : 이미지 1 x 28 x 28")
print(f"판별자 출력  : 숫자 1개 (클수록 진짜라고 판단)")
print()
print(f"생성자 파라미터: {sum(p.numel() for p in G.parameters()):,}개")
print(f"판별자 파라미터: {sum(p.numel() for p in D.parameters()):,}개")
print()
print("LeakyReLU를 쓰는 이유")
print("  ReLU는 음수 구간에서 그래디언트가 0이라 판별자 학습이 멈출 수 있다.")
print("  LeakyReLU(0.2)는 음수에서도 0.2배 기울기를 남긴다 (이론편 11.2절).")

---

## 2. GAN 학습 — 이론편 16.2절

학습은 **두 신경망을 번갈아** 갱신한다.

**판별자 차례**: 진짜는 1, 가짜는 0이라고 답하도록 학습
**생성자 차례**: 판별자가 자기 작품을 1이라 답하도록 학습

주의할 점이 하나 있다. 판별자를 학습할 때는 생성자까지 갱신되면 안 되므로
**`fake.detach()`로 그래프를 끊어야** 한다.

In [ ]:
import torch
import torch.nn as nn
import time

torch.manual_seed(42)
G = Generator().to(device)
D = Discriminator().to(device)

opt_G = torch.optim.Adam(G.parameters(), lr=2e-4)
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4)
criterion = nn.BCEWithLogitsLoss()

EPOCHS = 30
history = {"d_loss": [], "g_loss": [], "d_real": [], "d_fake": []}

print("=" * 60)
print("GAN 학습")
print("=" * 60)
t0 = time.time()

for epoch in range(EPOCHS):
    d_sum = g_sum = real_sum = fake_sum = 0.0
    n_batch = 0

    for real_imgs, _ in train_loader:
        real_imgs = real_imgs.to(device)
        b = len(real_imgs)
        ones = torch.ones(b, 1, device=device)
        zeros = torch.zeros(b, 1, device=device)

        # ── 1) 판별자 학습 ──
        z = torch.randn(b, LATENT_DIM, device=device)
        fake_imgs = G(z)

        opt_D.zero_grad()
        out_real = D(real_imgs)
        out_fake = D(fake_imgs.detach())      # 생성자로 그래디언트가 가지 않게
        d_loss = criterion(out_real, ones) + criterion(out_fake, zeros)
        d_loss.backward()
        opt_D.step()

        # ── 2) 생성자 학습 ──
        opt_G.zero_grad()
        out_fake2 = D(fake_imgs)              # 이번엔 detach 하지 않는다
        g_loss = criterion(out_fake2, ones)   # 판별자를 속이는 것이 목표
        g_loss.backward()
        opt_G.step()

        d_sum += d_loss.item(); g_sum += g_loss.item()
        real_sum += torch.sigmoid(out_real).mean().item()
        fake_sum += torch.sigmoid(out_fake).mean().item()
        n_batch += 1

    for k, v in zip(["d_loss", "g_loss", "d_real", "d_fake"],
                    [d_sum, g_sum, real_sum, fake_sum]):
        history[k].append(v / n_batch)

    if (epoch + 1) % 10 == 0:
        print(f"  에폭 {epoch+1:3}: D손실 {history['d_loss'][-1]:.3f}  "
              f"G손실 {history['g_loss'][-1]:.3f}  "
              f"판별자→진짜 {history['d_real'][-1]:.2f} / 가짜 {history['d_fake'][-1]:.2f}  "
              f"({time.time()-t0:.0f}초)")

print("-" * 60)
print(f"소요 시간: {time.time()-t0:.0f}초")

In [ ]:
import torch
import matplotlib.pyplot as plt

G.eval()
torch.manual_seed(7)
with torch.no_grad():
    z = torch.randn(16, LATENT_DIM, device=device)
    fake = G(z).cpu()

fig, axes = plt.subplots(2, 8, figsize=(13, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(fake[i].squeeze(), cmap="gray")
    ax.axis("off")
fig.suptitle("GAN이 생성한 이미지", fontsize=12)
plt.tight_layout()
plt.show()

print("18장의 VAE 결과와 비교해 보라.")
print("형태가 덜 정돈되었을 수 있지만, 가장자리는 더 또렷한 편이다.")
print()
print("작은 모델과 짧은 학습이라 품질에 한계가 있다.")
print("실제 GAN 결과물은 훨씬 크고 오래 학습한 것이다.")

---

## 3. 학습 불안정성 — 이론편 16.2절 ★

이론편 16.2절에서 "GAN은 학습이 불안정하다"고 했다. **왜 그런지 손실 곡선으로 확인한다.**

지금까지 본 모델들은 손실이 내려가면 좋은 것이었다. **GAN은 다르다.**
두 신경망이 서로 반대 방향을 향하므로, 어느 한쪽 손실이 낮다고 좋은 것이 아니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- 왼쪽: 두 손실 ---
ax = axes[0]
ax.plot(history["d_loss"], label="판별자 손실", linewidth=2, color="#1E40AF")
ax.plot(history["g_loss"], label="생성자 손실", linewidth=2, color="#EA580C")
ax.set_xlabel("에폭")
ax.set_ylabel("손실")
ax.set_title("두 손실은 서로 반대로 움직인다")
ax.legend()
ax.grid(alpha=0.3)

# --- 오른쪽: 판별자의 판단 ---
ax = axes[1]
ax.plot(history["d_real"], label="진짜를 진짜라 할 확률", linewidth=2, color="#0D9488")
ax.plot(history["d_fake"], label="가짜를 진짜라 할 확률", linewidth=2, color="#DC2626")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1.5)
ax.text(len(history["d_real"])*0.6, 0.52, "0.5 = 구별 못함 (이상적 균형)",
        fontsize=8, color="gray")
ax.set_xlabel("에폭")
ax.set_ylabel("확률")
ax.set_ylim(0, 1)
ax.set_title("판별자의 판단")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 60)
print("이 그래프를 읽는 법")
print("=" * 60)
print(f"  마지막 판별자 판단: 진짜 {history['d_real'][-1]:.2f} / 가짜 {history['d_fake'][-1]:.2f}")
print()
print("| 상황 | 뜻 |")
print("|------|-----|")
print("| 둘 다 0.5 근처       | 이상적. 판별자가 구별 못할 만큼 생성자가 잘함 |")
print("| 진짜1.0 / 가짜0.0    | 판별자 압승. 생성자가 배울 신호가 약함 |")
print("| 진짜0.5 / 가짜0.5    | 균형 |")
print()
print("손실이 낮다고 좋은 것이 아니라는 점이 GAN의 어려움이다.")
print("생성 결과를 눈으로 확인하는 것이 사실상 유일한 판단 기준이다.")

### 모드 붕괴 — 다양성이 사라지는 현상

이론편 16.2절에서 다룬 또 다른 문제다. 생성자가 **판별자를 잘 속이는 한두 가지만 계속 만들어 내는** 상황이다.

시험 문제를 푸는 학생이 "정답이 대체로 3장이더라"를 알아채고 전부 3번을 찍는 것과 비슷하다.
점수는 어느 정도 나오지만 실력은 늘지 않는다.

생성 이미지들이 얼마나 다양한지 측정해 보자.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

G.eval()
torch.manual_seed(0)
with torch.no_grad():
    z = torch.randn(200, LATENT_DIM, device=device)
    samples = G(z).cpu().view(200, -1).numpy()

# 생성물끼리 얼마나 다른지 — 서로 간 거리의 평균
from itertools import combinations
idx = np.random.RandomState(0).choice(200, 60, replace=False)
dists_fake = [np.linalg.norm(samples[i] - samples[j])
              for i, j in combinations(idx[:40], 2)]

# 실제 데이터끼리의 거리 (비교 기준)
real = torch.stack([train_data[i][0] for i in range(40)]).view(40, -1).numpy()
dists_real = [np.linalg.norm(real[i] - real[j]) for i, j in combinations(range(40), 2)]

print("=" * 55)
print("다양성 확인")
print("=" * 55)
print(f"{'':16}{'평균 거리':<16}{'표준편차'}")
print("-" * 55)
print(f"{'실제 데이터':16}{np.mean(dists_real):<16.3f}{np.std(dists_real):.3f}")
print(f"{'GAN 생성물':16}{np.mean(dists_fake):<16.3f}{np.std(dists_fake):.3f}")
print("-" * 55)
ratio = np.mean(dists_fake) / np.mean(dists_real)
print(f"비율: {ratio:.2f}")
print()
if ratio < 0.6:
    print("생성물끼리 서로 비슷하다 — 모드 붕괴 징후")
else:
    print("어느 정도 다양성이 유지되고 있다")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(dists_real, bins=25, alpha=0.6, label="실제 데이터", color="#1E40AF")
ax.hist(dists_fake, bins=25, alpha=0.6, label="GAN 생성물", color="#EA580C")
ax.set_xlabel("이미지 쌍 사이의 거리")
ax.set_ylabel("빈도")
ax.set_title("다양성 비교")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## 4. 노이즈 누적 — 이론편 16.3절 값 검증 ★

Diffusion은 완전히 다른 발상이다. **한 번에 만들지 않고, 노이즈를 조금씩 걷어낸다.**

먼저 그 반대 방향인 **정방향 과정**을 이해해야 한다. 원본에 노이즈를 조금씩 더해
결국 순수한 노이즈로 만드는 과정이다.

$$x_t = \sqrt{1-\beta}\;x_{t-1} + \sqrt{\beta}\;\epsilon$$

한 단계에 더하는 노이즈는 아주 적다($\beta = 0.02$). 그런데 반복하면 어떻게 될까.

**이론편 16.3절에서 계산한 표**

| $t$ | $\bar\alpha_t$ | 원본 비중 | 노이즈 비중 |
|---|---|---|---|
| 1 | 0.980 | 0.990 | 0.141 |
| 10 | 0.817 | 0.904 | 0.428 |
| 50 | 0.364 | 0.603 | 0.797 |
| 100 | 0.133 | 0.364 | 0.931 |
| 500 | 0.00004 | 0.006 | 1.000 |

In [ ]:
import numpy as np

BETA = 0.02

print("=" * 65)
print("이론편 16.3절 노이즈 누적표 검증")
print("=" * 65)
print(f"beta = {BETA} (매 단계 섞는 노이즈 비율)")
print(f"alpha = 1 - beta = {1-BETA}")
print()
print(f"{'t':<8}{'alpha_bar':<16}{'원본 비중':<16}{'노이즈 비중':<16}{'이론편'}")
print("-" * 65)

book = {1: (0.990, 0.141), 10: (0.904, 0.428), 50: (0.603, 0.797),
        100: (0.364, 0.931), 500: (0.006, 1.000)}

for t in [1, 10, 50, 100, 500]:
    alpha_bar = (1 - BETA) ** t          # 누적 곱
    keep = np.sqrt(alpha_bar)            # 원본이 남는 비율
    noise = np.sqrt(1 - alpha_bar)       # 노이즈 비율
    bk, bn = book[t]
    print(f"{t:<8}{alpha_bar:<16.5f}{keep:<16.4f}{noise:<16.4f}{bk:.3f}/{bn:.3f}")
    assert abs(keep - bk) < 0.002, f"t={t}에서 이론편 값과 다릅니다"
    assert abs(noise - bn) < 0.002

print("-" * 65)
print("[OK] 이론편 16.3절 표와 일치")
print()
print("50단계쯤에서 원본과 노이즈의 비중이 뒤집힌다.")
print("500단계에서는 원본이 사실상 사라진다.")
print()
print("이론편 4.3절 거듭제곱의 위력이 여기서도 작동한다 —")
print("0.98처럼 1에 가까운 값도 수백 번 곱하면 0에 수렴한다.")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 실제 이미지로 확인
img = train_data[0][0]
torch.manual_seed(0)
noise = torch.randn_like(img)

steps = [0, 10, 50, 100, 200, 500]
fig, axes = plt.subplots(1, len(steps), figsize=(14, 2.6))

for ax, t in zip(axes, steps):
    ab = (1 - BETA) ** t
    noisy = np.sqrt(ab) * img + np.sqrt(1 - ab) * noise
    ax.imshow(noisy.squeeze(), cmap="gray", vmin=-1, vmax=1.5)
    ax.set_title(f"t={t}\n원본 {np.sqrt(ab):.2f}", fontsize=8)
    ax.axis("off")

fig.suptitle("노이즈가 쌓여 가는 과정 (정방향)", fontsize=12)
plt.tight_layout()
plt.show()

print("각 단계는 '거의 같은 이미지에서 아주 조금 노이즈를 걷어내기'라는 쉬운 문제다.")
print()
print("신경망이 배워야 할 것은 한 번에 다 되돌리는 어려운 일이 아니라,")
print("조금씩 걷어내는 일을 정확히 하는 것뿐이다 (이론편 16.3절).")

---

## 5. Diffusion 직접 구현 — 이론편 16.3절

이제 노이즈를 걷어내는 신경망을 만든다. 학습 목표는 놀랍도록 단순하다.

$$L = \big\lVert \epsilon - \epsilon_\theta(x_t, t) \big\rVert^2$$

**"이 노이즈 낀 이미지에 섞여 있던 노이즈가 무엇이었는지 맞혀라."**
복잡한 확률분포를 다루는 대신 평범한 회귀 문제(이론편 8.2절)가 된다.

학습 절차는 이렇다.

1. 원본 이미지 $x_0$를 하나 고른다
2. 시점 $t$를 무작위로 고른다
3. 노이즈 $\epsilon$을 만들어 $x_t$를 계산한다
4. 신경망이 $\epsilon$을 맞히도록 학습한다

In [ ]:
import torch
import torch.nn as nn

T = 200        # 전체 단계 수 (실제 모델은 1000 이상)

# 노이즈 일정(schedule) — 처음엔 조금, 나중엔 많이
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)    # 누적 곱

print("=" * 55)
print("노이즈 일정")
print("=" * 55)
print(f"단계 수 T = {T}")
print(f"beta 범위 : {betas[0]:.5f} ~ {betas[-1]:.5f}")
print()
print(f"{'t':<8}{'beta':<14}{'alpha_bar':<16}{'원본 비중'}")
print("-" * 55)
for t in [0, 50, 100, 150, 199]:
    print(f"{t:<8}{betas[t]:<14.5f}{alpha_bars[t]:<16.5f}{alpha_bars[t].sqrt():.4f}")
print("-" * 55)
print()
print("beta를 일정하게 두지 않고 점점 키우는 이유:")
print("  초반에는 조금씩 흐리게 해야 세부 정보가 천천히 사라진다.")
print("  후반에는 어차피 형태가 없으므로 크게 더해도 무방하다.")


class SimpleDenoiser(nn.Module):
    # 노이즈를 예측하는 신경망 (이론편 16.3절)
    #
    # 시점 t를 함께 입력하는 이유:
    #   같은 이미지라도 '얼마나 노이즈가 낀 상태인가'에 따라
    #   걷어내야 할 양이 다르기 때문이다.

    def __init__(self, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784 + 1, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 784),
        )

    def forward(self, x, t):
        # 시점을 0~1 범위로 정규화해 함께 넣는다
        t_norm = t.float().unsqueeze(1) / T
        h = torch.cat([x.flatten(1), t_norm], dim=1)
        return self.net(h)


denoiser = SimpleDenoiser().to(device)
print()
print(f"파라미터: {sum(p.numel() for p in denoiser.parameters()):,}개")

In [ ]:
import torch
import torch.nn as nn
import time

torch.manual_seed(42)
denoiser = SimpleDenoiser().to(device)
optimizer = torch.optim.Adam(denoiser.parameters(), lr=1e-3)

alpha_bars_dev = alpha_bars.to(device)

EPOCHS = 40
losses_diff = []

print("=" * 60)
print("Diffusion 학습 — 노이즈 맞히기")
print("=" * 60)
t0 = time.time()

for epoch in range(EPOCHS):
    total, n = 0.0, 0
    for x0, _ in train_loader:
        x0 = x0.to(device)
        b = len(x0)

        # 1) 시점을 무작위로 고른다
        t = torch.randint(0, T, (b,), device=device)

        # 2) 노이즈를 만들어 x_t 를 계산한다
        noise = torch.randn_like(x0)
        ab = alpha_bars_dev[t].view(-1, 1, 1, 1)
        x_t = ab.sqrt() * x0 + (1 - ab).sqrt() * noise

        # 3) 신경망이 그 노이즈를 맞히도록 학습
        optimizer.zero_grad()
        pred_noise = denoiser(x_t, t)
        loss = nn.functional.mse_loss(pred_noise, noise.flatten(1))
        loss.backward()
        optimizer.step()

        total += loss.item() * b
        n += b

    losses_diff.append(total / n)
    if (epoch + 1) % 10 == 0:
        print(f"  에폭 {epoch+1:3}: 손실 {losses_diff[-1]:.4f}  ({time.time()-t0:.0f}초)")

print("-" * 60)
print(f"소요 시간: {time.time()-t0:.0f}초")
print()
print("손실이 1.0 근처에서 시작하는 이유:")
print("  아무것도 모르면 0을 예측하는데, 표준정규 노이즈의 분산이 1이므로")
print("  MSE가 대략 1이 된다. 그보다 낮아지면 뭔가 배운 것이다.")

### 역과정 — 노이즈에서 이미지로

학습이 끝났으면 **순수한 노이즈에서 시작해 한 단계씩 걷어낸다.**

각 단계에서 신경망이 "여기 섞인 노이즈는 이것"이라고 답하면, 그만큼 빼고
조금 덜 노이즈 낀 상태로 옮겨간다. 이를 $T$번 반복한다.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt


@torch.no_grad()
def sample_diffusion(model, n_samples=8, record_steps=None):
    # 순수 노이즈에서 시작해 한 단계씩 되돌린다 (이론편 16.3절 역과정)
    model.eval()
    x = torch.randn(n_samples, 1, 28, 28, device=device)
    snapshots = {}

    for t in reversed(range(T)):
        t_batch = torch.full((n_samples,), t, device=device, dtype=torch.long)
        pred_noise = model(x, t_batch).view(-1, 1, 28, 28)

        a = alphas[t].to(device)
        ab = alpha_bars[t].to(device)

        # 예측한 노이즈를 빼서 한 단계 되돌린다
        x = (x - (1 - a) / (1 - ab).sqrt() * pred_noise) / a.sqrt()

        # 마지막 단계가 아니면 약간의 노이즈를 다시 더한다
        if t > 0:
            x = x + betas[t].to(device).sqrt() * torch.randn_like(x)

        if record_steps and t in record_steps:
            snapshots[t] = x.clone().cpu()

    return x.cpu(), snapshots


print("생성 중... (200단계를 거슬러 올라간다)")
record = [199, 150, 100, 50, 20, 0]
generated, snaps = sample_diffusion(denoiser, n_samples=8, record_steps=record)

# 생성 과정 보여주기
fig, axes = plt.subplots(1, len(record), figsize=(14, 2.6))
for ax, t in zip(axes, record):
    if t in snaps:
        ax.imshow(snaps[t][0].squeeze(), cmap="gray")
        ax.set_title(f"t={t}", fontsize=9)
    ax.axis("off")
fig.suptitle("노이즈에서 이미지로 (역과정)", fontsize=12)
plt.tight_layout()
plt.show()

# 최종 결과
fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for i, ax in enumerate(axes):
    ax.imshow(generated[i].squeeze(), cmap="gray")
    ax.axis("off")
fig.suptitle("Diffusion이 생성한 이미지", fontsize=12)
plt.tight_layout()
plt.show()

print("작은 모델과 짧은 학습이라 품질에 한계가 있다.")
print("실제 Diffusion 모델은 U-Net 구조에 T=1000 이상을 쓴다 (이론편 16.4절).")
print()
print("여기서 확인할 것은 '노이즈에서 형태가 서서히 나타난다'는 원리다.")

---

## 6. 세 모델 비교 — 이론편 16.4절

VAE(18장), GAN, Diffusion을 같은 데이터로 만들어 비교한다.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

print("=" * 68)
print("생성 모델 비교 (이론편 16.4절)")
print("=" * 68)
print(f"{'':14}{'생성 방식':<22}{'속도':<12}{'주요 문제'}")
print("-" * 68)
print(f"{'VAE':14}{'잠재 벡터 → 디코더':<22}{'1회':<12}{'흐릿함'}")
print(f"{'GAN':14}{'무작위 → 생성자':<22}{'1회':<12}{'학습 불안정·모드 붕괴'}")
print(f"{'Diffusion':14}{'노이즈 → T번 반복':<22}{f'{T}회':<12}{'생성이 느림'}")
print("-" * 68)
print()

# 생성 속도 측정
import time

G.eval()
torch.manual_seed(0)
t0 = time.time()
with torch.no_grad():
    _ = G(torch.randn(8, LATENT_DIM, device=device))
gan_time = time.time() - t0

t0 = time.time()
_, _ = sample_diffusion(denoiser, n_samples=8)
diff_time = time.time() - t0

print("이미지 8장 생성 시간")
print(f"  GAN       : {gan_time*1000:8.1f} ms  (신경망 1회 통과)")
print(f"  Diffusion : {diff_time*1000:8.1f} ms  (신경망 {T}회 통과)")
_r = f"{diff_time/gan_time:8.0f}배" if gan_time > 0 else "   측정 불가"
print(f"  차이      : {_r}")
print()
print("Diffusion이 느린 이유가 명확하다 — 단계마다 신경망을 부른다.")
print("품질을 얻는 대가로 속도를 내준 셈이다 (이론편 16.4절).")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 선명도 비교 (18장에서 쓴 방법)
def sharpness(imgs):
    x = imgs.detach().cpu().squeeze().numpy()
    dx = np.abs(np.diff(x, axis=-1)).mean()
    dy = np.abs(np.diff(x, axis=-2)).mean()
    return (dx + dy) / 2

real_imgs = torch.stack([train_data[i][0] for i in range(8)])

torch.manual_seed(0)
G.eval()
with torch.no_grad():
    gan_imgs = G(torch.randn(8, LATENT_DIM, device=device)).cpu()
diff_imgs, _ = sample_diffusion(denoiser, n_samples=8)

print("=" * 55)
print("선명도 비교 (인접 픽셀 차이)")
print("=" * 55)
s_real = sharpness(real_imgs)
for name, imgs in [("실제 데이터", real_imgs), ("GAN", gan_imgs),
                   ("Diffusion", diff_imgs)]:
    s = sharpness(imgs)
    print(f"  {name:<14}{s:.4f}   (실제 대비 {s/s_real*100:5.0f}%)")
print("-" * 55)
print()
print("18장의 VAE 결과와도 비교해 보면 좋다.")
print("다만 이 수치만으로 품질을 판단할 수는 없다 —")
print("노이즈가 많아도 이 값은 높게 나오기 때문이다.")

fig, axes = plt.subplots(3, 8, figsize=(13, 5))
for row, (name, imgs) in enumerate([("실제", real_imgs), ("GAN", gan_imgs),
                                     ("Diffusion", diff_imgs)]):
    for col in range(8):
        axes[row, col].imshow(imgs[col].detach().squeeze(), cmap="gray")
        axes[row, col].axis("off")
    axes[row, 0].set_title(name, loc="left", fontsize=10)
plt.tight_layout()
plt.show()

---

## 7. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| **16.3** | **노이즈 누적표 5개 시점** | **일치** ✓ |
| 16.2 | GAN 학습 불안정 | 손실 곡선으로 확인 ✓ |
| 16.2 | 모드 붕괴 | 다양성 측정 ✓ |
| 16.3 | 각 단계는 쉬운 문제 | 시각화 확인 ✓ |
| 16.5 | Diffusion이 느림 | 속도 측정 ✓ |

### 세 모델의 발상 차이

| 모델 | 핵심 발상 |
|---|---|
| VAE | 압축했다 되돌리되, 잠재 공간을 정규분포로 정리 |
| GAN | 판별자에게 평가받게 해서 "진짜처럼" 만들기 |
| Diffusion | 어려운 일을 쉬운 단계로 쪼개기 |

### 기억할 것

| 항목 | 요점 |
|---|---|
| `detach()` | 판별자 학습 시 생성자로 그래디언트가 가지 않게 |
| LeakyReLU | 판별자의 그래디언트가 죽지 않도록 |
| GAN 손실 | 낮다고 좋은 것이 아님 — 눈으로 확인해야 함 |
| Diffusion 학습 | "섞인 노이즈 맞히기"라는 단순 회귀 |
| 시점 t 입력 | 노이즈 정도에 따라 걷어낼 양이 다르므로 |
| 속도 | GAN 1회 vs Diffusion T회 |

### Part 3을 마치며

15~19장에서 딥러닝의 주요 구조를 다뤘다.

| 장 | 다룬 것 |
|---|---|
| 11 | CNN — 공간 정보 |
| 12 | RNN/LSTM — 순서 정보 |
| 13 | 강화학습 — 보상으로 배우기 |
| 14 | VAE — 잠재 공간과 생성 |
| 15 | GAN·Diffusion — 생성의 다른 접근 |

### 다음 장

**20. Attention 직접 구현 ★** — 4부가 시작된다.
16장에서 본 RNN의 한계(순차 처리, 정보 병목)를 정면으로 해결한 구조다.

**이론편 18.3절에서 손으로 계산한 Attention 가중치 (0.401, 0.198, 0.401)를 검증**한다.
1·2권 연결의 가장 중요한 지점 중 하나다.